In [ ]:
#COMBINED MULTIMODAL LOGISTIC REFRESSION MODEL (Age + Cognition + GFAP + NEFL)

"""
This script runs a Single combined model of incident dementia on the Olink baseline
sample:
    dem ~ age + composite_cognition + GFAP + NEFL      (all per 1 SD)
plus nested reference models for the ROC (age, age+cog, age+GFAP, age+NEFL).
Run from 06_Biomarkers_correlation/Olink/ (same paths as cog_biomarker_comp).
Outputs: one OR forest + one ROC figure.
"""

import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.transforms import blended_transform_factory
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve

GROUP = 'hc'                     # 'all' | 'hc' | 'ic'
output_root = f'combined_multimodal_{GROUP}'
os.makedirs(output_root, exist_ok=True)
biomarkers = ['gfap', 'nefl']

COG_PATH = 'cv_results_all_tests_merged_with_groups.csv'
BIO_PATH = 'stage1_adjusted_data.csv'
COL_EID, COL_TEST, COL_Z = 'eid', 'test_name', 'z_score'
COL_AGE, COL_DEM = 'age_at_test', 'dem_all'
COMPOSITE_TESTS = ['FI', 'NM', 'PMat', 'RT']
DEFICIT_TESTS   = ['RT', 'PMat']          # higher = worse within composite

# viridis -> magma palette (matches the figures notebook)
_vir = plt.get_cmap('viridis')(np.linspace(0.00, 0.90, 128))
_mag = plt.get_cmap('magma')(np.linspace(0.90, 0.45, 128))
RAMP = ListedColormap(np.vstack([_vir, _mag]))
_b = np.array(RAMP(0.86)[:3])
COL = {'age': (0.45, 0.45, 0.45), 'cognition': RAMP(0.18)[:3],
       'gfap': tuple(_b * 0.58 + 0.42), 'nefl': tuple(_b * 0.82), 'all': 'black'}

SAVE = True
def save_fig(fig, name):
    if not SAVE: return
    for ext in ('svg', 'png'):
        fig.savefig(os.path.join(output_root, f'{name}.{ext}'), dpi=300,
                    bbox_inches='tight', facecolor='white')
    print('saved', name)

In [ ]:
#LOAD + BUILD ANALYTIC SAMPLE (one row per person: dem, age, cog, gfap, nefl)

cog = pd.read_csv(COG_PATH)
bio = pd.read_csv(BIO_PATH)
COL_INST = 'instance' if 'instance' in cog.columns else None

bio_keep = [COL_EID] + biomarkers + [c for c in ['hc', 'ic'] if c in bio.columns]
bio_person = (bio[bio_keep].dropna(subset=biomarkers, how='all')
              .groupby(COL_EID, as_index=False).first())
df = cog.merge(bio_person, on=COL_EID, how='inner', suffixes=('', '_bio'))
if COL_INST is not None:
    df = df[df[COL_INST] == 0].copy()          # contemporaneous (baseline) visit
if GROUP != 'all':
    df = df[df[GROUP] == 1].copy()

# composite cognition = mean of direction-aligned baseline tests (higher = worse)
c = df[df[COL_TEST].isin(COMPOSITE_TESTS)].copy()
c['zc'] = np.where(c[COL_TEST].isin(DEFICIT_TESTS), -c[COL_Z], c[COL_Z])   # higher=better
agg = {'cog': ('zc', 'mean'), COL_AGE: (COL_AGE, 'mean'), COL_DEM: (COL_DEM, 'first')}
for b in biomarkers:
    agg[b] = (b, 'first')
dat = c.groupby(COL_EID).agg(**agg).reset_index()
dat['cog'] = -dat['cog']                        # flip -> higher = worse (match markers)
dat = dat[[COL_EID, COL_DEM, COL_AGE, 'cog', 'gfap', 'nefl']].dropna()

# standardise every predictor to SD units (OR = per 1 SD)
for col in ['cog', 'gfap', 'nefl', COL_AGE]:
    dat[col + '_z'] = (dat[col] - dat[col].mean()) / dat[col].std(ddof=0)

n, events = len(dat), int(dat[COL_DEM].sum())
print(f"GROUP={GROUP}: n={n} people, events={events}, EPV={events/4:.1f}")

In [ ]:
#FIT — nested models + combined multimodal

A, ZC, G, F, Y = COL_AGE + '_z', 'cog_z', 'gfap_z', 'nefl_z', COL_DEM
def fit(f): return smf.glm(f, data=dat, family=sm.families.Binomial()).fit()
models = {
    'age':  fit(f'{Y} ~ {A}'),
    'cog':  fit(f'{Y} ~ {A} + {ZC}'),
    'nefl': fit(f'{Y} ~ {A} + {F}'),
    'gfap': fit(f'{Y} ~ {A} + {G}'),
    'all':  fit(f'{Y} ~ {A} + {ZC} + {G} + {F}'),
}
Mall = models['all']

# OR-per-SD table from the combined model
or_rows = []
for term, name in [(A, 'age'), (ZC, 'cognition'), (G, 'gfap'), (F, 'nefl')]:
    ci = Mall.conf_int().loc[term]
    or_rows.append({'predictor': name, 'or': np.exp(Mall.params[term]),
                    'or_lo': np.exp(ci[0]), 'or_hi': np.exp(ci[1]), 'p': Mall.pvalues[term]})
or_df = pd.DataFrame(or_rows)

def nagelkerke(m, n):
    cox = 1 - np.exp(2 * (m.llnull - m.llf) / n)
    return cox / (1 - np.exp(2 * m.llnull / n))
auc = {k: roc_auc_score(dat[Y], m.predict(dat)) for k, m in models.items()}
nag = {k: nagelkerke(m, n) for k, m in models.items()}

or_df.to_csv(os.path.join(output_root, 'combined_or.csv'), index=False)
print("OR per 1 SD (combined model, mutually adjusted):")
print(or_df.round(3).to_string(index=False))
print("\nAUC:", {k: round(v, 3) for k, v in auc.items()})
print("Nagelkerke R2:", {k: round(v, 3) for k, v in nag.items()})

In [ ]:
#EXPORT for the visualisation notebook
def tjur(m):
    p = m.predict(dat)
    return p[dat[Y] == 1].mean() - p[dat[Y] == 0].mean()

pd.DataFrame([{'model': k, 'auc': auc[k], 'nagelkerke': nag[k], 'tjur': tjur(m)}
              for k, m in models.items()]
).to_csv(os.path.join(output_root, 'combined_fit_metrics.csv'), index=False)

roc_rows = []
for k, m in models.items():
    fpr, tpr, _ = roc_curve(dat[Y], m.predict(dat))
    roc_rows.append(pd.DataFrame({'model': k, 'fpr': fpr, 'tpr': tpr}))
pd.concat(roc_rows, ignore_index=True).to_csv(
    os.path.join(output_root, 'combined_roc_points.csv'), index=False)

print('exported combined_fit_metrics.csv, combined_roc_points.csv')

In [ ]:
#FOREST — ORs from the combined model (age, cognition, GFAP, NEFL)

ORDER = ['age', 'cognition', 'gfap', 'nefl']
LABEL = {'age': 'Age', 'cognition': 'Cognition', 'gfap': 'GFAP', 'nefl': 'NEFL'}
d = or_df.set_index('predictor')
def fp(p): return 'p<0.001' if p < 0.001 else f'p={p:.3f}'

ROW, PAD = 0.32, 0.22
ypos = {nm: -i * ROW for i, nm in enumerate(ORDER)}
top0 = max(ypos.values()) + PAD; ylo = min(ypos.values()) - PAD

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.axvline(1, color='0.35', ls='--', lw=1, zorder=2)
tr = blended_transform_factory(ax.transAxes, ax.transData)
for nm in ORDER:
    if nm not in d.index: continue
    r = d.loc[nm]; y = ypos[nm]; c = COL[nm]
    ax.plot([r['or_lo'], r['or_hi']], [y, y], color=c, lw=2.6, solid_capstyle='round', zorder=3)
    ax.scatter([r['or']], [y], s=70, color=c, edgecolors=c, zorder=4)
    if r['p'] < 0.05:
        ax.text(r['or_hi'] + 0.03, y, '*', color=c, fontsize=14, va='center', fontweight='bold')
    ax.text(-0.02, y, LABEL[nm], transform=tr, fontsize=11, va='center', ha='right',
            fontweight='bold', color=c)
    ax.text(1.02, y, fp(r['p']), transform=tr, fontsize=8.5, va='center',
            color='0.15' if r['p'] < 0.05 else '0.55')
ax.text(1.02, top0, 'p-value', transform=tr, fontsize=9, fontweight='bold', va='bottom')
ax.set_title('Combined model — effects on incident dementia', fontsize=13, pad=10)
ax.set_ylim(ylo, top0 + 0.05); ax.set_yticks([])
ax.set_xlabel('Odds ratio per 1 SD (mutually adjusted: age, cognition, GFAP, NEFL)')
ax.spines[['top', 'right', 'left']].set_visible(False); ax.tick_params(length=0)
for xt in ax.get_xticks():
    ax.vlines(xt, ylo, top0 + 0.05, color='0.93', lw=0.7, zorder=0.5)
fig.subplots_adjust(left=0.18, right=0.82, top=0.88, bottom=0.20)
plt.show(); save_fig(fig, 'combined_forest')

In [ ]:
#ROC — Age / Age+cog / Age+NEFL / Age+GFAP / multimodal

STYLE = [('age',  'Age',              COL['age']),
         ('cog',  'Age + cognition',  COL['cognition']),
         ('nefl', 'Age + NEFL',       COL['nefl']),
         ('gfap', 'Age + GFAP',       COL['gfap']),
         ('all',  'Multimodal (all)', COL['all'])]

fig, ax = plt.subplots(figsize=(6.2, 6.2))
def draw(target, legend=False):
    for key, label, c in STYLE:
        fpr, tpr, _ = roc_curve(dat[Y], models[key].predict(dat))
        lw = 1.6 if key == 'all' else 1
        lbl = f'{label} (AUC {auc[key]:.3f}, R² {nag[key]:.3f})' if legend else None
        target.plot(fpr, tpr, color=c, lw=lw, label=lbl, zorder=4 if key == 'all' else 3)

draw(ax, legend=True)
ax.plot([0, 1], [0, 1], color='0.8', ls='--', lw=1)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('Combined multimodal discrimination (incident dementia)', fontsize=13, pad=10)
ax.legend(loc='lower right', frameon=False, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# zoomed inset on the separation elbow
axins = ax.inset_axes([0.470, 0.25, 0.5, 0.5])
draw(axins, legend=False)
axins.plot([0, 1], [0, 1], color='0.85', ls='--', lw=0.8)
axins.set_xlim(0.2, 0.30); axins.set_ylim(0.7, 0.8)
axins.set_xticks([0.2, 0.25, 0.3]); axins.set_yticks([0.7, 0.75, 0.8])
axins.tick_params(labelsize=7, length=2)
for sp in axins.spines.values(): sp.set_color('0.5')
ax.indicate_inset_zoom(axins, edgecolor='0.55', lw=0.8, alpha=0.7)

fig.subplots_adjust(left=0.11, right=0.97, top=0.92, bottom=0.10)
plt.show(); save_fig(fig, 'combined_roc')